# XLE Test-Time Feature-Noise Robustness

This notebook extends the first controlled test-time input-noise experiment for the completed XLE Logistic Regression and Random Forest clean baselines. Both models are fitted once on clean training data. Additive Gaussian noise is introduced only into selected fixed-test predictors, and neither targets nor model parameters are altered during the experiment. Intensities 0.05–0.20 are the original moderate-noise pilot range. Intensities 0.50–1.00 are an exploratory stress-test extension added after inspection of the pilot's relatively small degradation; this post-pilot motivation is disclosed rather than presented as pre-specified.

In [ ]:
from pathlib import Path
import os

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the cloned repository.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.venv' / '.matplotlib'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'xle_feature_dataset.csv'
DICTIONARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xlk_feature_dictionary.csv'
LOGISTIC_METRICS_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xle_logistic_baseline_metrics.csv'
RF_METRICS_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xle_random_forest_test_metrics.csv'
DETAILED_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xle_noise_robustness_detailed.csv'
SUMMARY_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xle_noise_robustness_summary.csv'
CLIPPING_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'xle_noise_clipping_diagnostics.csv'
FIGURE_PATHS = {
    'balanced_accuracy': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_balanced_accuracy_curves.png',
    'f1_score': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_f1_curves.png',
    'roc_auc': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_roc_auc_curves.png',
    'average_precision': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_average_precision_curves.png',
}
DEGRADATION_FIGURE_PATHS = {
    'balanced_accuracy': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_balanced_accuracy_degradation.png',
    'f1_score': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_f1_degradation.png',
    'roc_auc': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_roc_auc_degradation.png',
    'average_precision': PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_average_precision_degradation.png',
}
HEATMAP_PATH = PROJECT_ROOT / 'outputs' / 'figures' / 'xle_noise_degradation_heatmap_intensity_1_00.png'

for path in [DICTIONARY_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f'Required project input is missing: {path}')
sns.set_theme(style='whitegrid')
print(f'Confirmed project root: {PROJECT_ROOT}')

## 1. Preserved data, targets and experimental conditions

The XLE raw data, eleven predictors, target, chronological 80/20 split and five-observation purge are constructed independently in this notebook using the same methodology as XLK. XLE training data independently determine the target threshold and feature-noise scales. The fixed classification threshold remains 0.5. Existing XLE outputs are read before recomputation so that all numerical results can be checked for invariance.

In [ ]:
import yfinance as yf
raw_path = PROJECT_ROOT / "data/raw/xle_daily_2000_2026.csv"
if raw_path.is_file():
    raw_download = pd.read_csv(raw_path, header=[0, 1], index_col=0, parse_dates=True)
    print('Using the existing local XLE raw file; no new download was made.')
else:
    raw_download = yf.download("XLE", start="2000-01-03", end="2026-07-01", auto_adjust=False, progress=False)
    assert not raw_download.empty
    raw_download.to_csv(raw_path)
if isinstance(raw_download.columns, pd.MultiIndex):
    assert "XLE" in set(raw_download.columns.get_level_values(-1))
    raw = raw_download.xs("XLE", axis=1, level=-1).copy()
else:
    raw = raw_download.copy()
raw.index = pd.to_datetime(raw.index); raw.index.name="Date"
required={"Open","High","Low","Close","Adj Close","Volume"}
assert required.issubset(raw.columns) and raw.index.is_monotonic_increasing and raw.index.is_unique
assert (raw[["Open","High","Low","Close","Adj Close"]]>0).all().all()
lr=np.log(raw["Adj Close"]/raw["Adj Close"].shift(1))
rv=np.sqrt(pd.concat([lr.shift(-k).pow(2) for k in range(1,6)],axis=1).sum(axis=1,min_count=5))
v20=lr.rolling(20).std(ddof=1)*np.sqrt(252)
base=pd.DataFrame({"future_rv_5d":rv,"v20":v20}).dropna()
cut=int(np.floor(.8*len(base))); provisional=base.iloc[:cut]; test_base=base.iloc[cut:]
first_test=test_base.index[0]; purged_dates=provisional.index[-5:].copy(); train_base=provisional.iloc[:-5]
assert len(purged_dates)==5
assert first_test == pd.Timestamp('2021-03-10')
assert (purged_dates < first_test).all()
for purged_date in purged_dates:
    purged_position = raw.index.get_loc(purged_date)
    future_window_dates = raw.index[purged_position + 1:purged_position + 6]
    assert len(future_window_dates) == 5 and (future_window_dates >= first_test).any()
threshold=train_base.future_rv_5d.quantile(.75)
targets=pd.concat([train_base.assign(high_volatility=lambda x:(x.future_rv_5d>threshold).astype(int),sample_period="train"),
                   test_base.assign(high_volatility=lambda x:(x.future_rv_5d>threshold).astype(int),sample_period="test")])
f=pd.DataFrame(index=raw.index); f["log_return_1d"]=lr; f["return_5d"]=lr.rolling(5).sum(); f["return_20d"]=lr.rolling(20).sum()
f["volatility_5d"]=lr.rolling(5).std(ddof=1)*np.sqrt(252); f["volatility_20d"]=v20
f["downside_volatility_20d"]=np.sqrt(lr.clip(upper=0).pow(2).rolling(20).mean())*np.sqrt(252)
f["price_to_ma_10"]=raw["Adj Close"]/raw["Adj Close"].rolling(10).mean()-1
f["price_to_ma_50"]=raw["Adj Close"]/raw["Adj Close"].rolling(50).mean()-1
d=raw["Adj Close"].diff(); ag=d.clip(lower=0).rolling(14).mean(); al=(-d.clip(upper=0)).rolling(14).mean()
f["rsi_14"]=(100-100/(1+ag/al.replace(0,np.nan))).mask((al==0)&(ag>0),100).mask((al==0)&(ag==0),50)
assert (raw["Close"]>0).all() and (raw["Volume"].rolling(20).mean().dropna()>0).all()
f["volume_ratio_20d"]=raw["Volume"]/raw["Volume"].rolling(20).mean()
f["intraday_range"]=(raw["High"]-raw["Low"])/raw["Close"]
PREDICTORS=["log_return_1d","return_5d","return_20d","volatility_5d","volatility_20d","downside_volatility_20d","price_to_ma_10","price_to_ma_50","rsi_14","volume_ratio_20d","intraday_range"]
out=targets[["future_rv_5d","high_volatility","sample_period"]].join(f[PREDICTORS]).dropna().reset_index()
assert not out.Date.isin(purged_dates).any() and out[PREDICTORS].notna().all().all()
out.to_csv(PROJECT_ROOT/"data/processed/xle_feature_dataset.csv",index=False)
print("XLE range",raw.index.min(),raw.index.max(),"raw",len(raw),"train/test",out.sample_period.value_counts().to_dict(),"threshold",threshold,"purged",list(purged_dates))


In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=['Date'])
feature_dictionary = pd.read_csv(DICTIONARY_PATH)
logistic_reference = None
rf_reference = None
existing_detailed_reference = pd.read_csv(DETAILED_PATH) if DETAILED_PATH.exists() else None
existing_summary_reference = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else None
existing_clipping_reference = pd.read_csv(CLIPPING_PATH) if CLIPPING_PATH.exists() else None
CLEAN_METRICS_PATH = PROJECT_ROOT / 'outputs/tables/xle_clean_baseline_metrics.csv'
COMPARISON_PATH = PROJECT_ROOT / 'outputs/tables/xlk_xle_key_results_comparison.csv'
existing_clean_reference = pd.read_csv(CLEAN_METRICS_PATH) if CLEAN_METRICS_PATH.exists() else None
existing_comparison_reference = pd.read_csv(COMPARISON_PATH) if COMPARISON_PATH.exists() else None
predictors = feature_dictionary['feature_name'].tolist()
target = 'high_volatility'

FEATURE_GROUPS = {
    'returns': ['log_return_1d', 'return_5d', 'return_20d'],
    'volatility': ['volatility_5d', 'volatility_20d', 'downside_volatility_20d'],
    'momentum': ['price_to_ma_10', 'price_to_ma_50', 'rsi_14'],
    'volume_and_range': ['volume_ratio_20d', 'intraday_range'],
    'all_predictors': predictors.copy(),
}
PILOT_INTENSITIES = [0.05, 0.10, 0.20]
STRESS_TEST_INTENSITIES = [0.50, 1.00]
INTENSITIES = [*PILOT_INTENSITIES, *STRESS_TEST_INTENSITIES]
SEEDS = list(range(30))
METRICS = [
    'accuracy', 'balanced_accuracy', 'precision', 'recall',
    'f1_score', 'roc_auc', 'average_precision',
]

assert len(predictors) == 11 and set(FEATURE_GROUPS['all_predictors']) == set(predictors), \
    'Exactly the 11 documented predictors are required.'
assert data['Date'].is_monotonic_increasing and data['Date'].is_unique, 'Dates must be increasing and unique.'
assert not data[predictors + [target, 'future_rv_5d', 'sample_period']].isna().any().any(), \
    'Predictors, targets and assignments must be complete.'
assert np.isfinite(data[predictors].to_numpy(dtype=float)).all(), 'Predictors must be finite.'

original_dates = data[['Date', 'sample_period']].copy(deep=True)
original_targets = data[['Date', target]].copy(deep=True)
train_data = data.loc[data['sample_period'].eq('train')].copy()
test_data = data.loc[data['sample_period'].eq('test')].copy()
X_train = train_data[predictors].copy()
y_train = train_data[target].astype(int).copy()
X_test_clean = test_data[predictors].copy()
y_test = test_data[target].astype(int).copy()
test_targets_before_experiment = y_test.copy(deep=True)
training_standard_deviation = X_train.std(ddof=1)

assert len(purged_dates) == 5, 'Exactly five XLE observations must be purged.'
assert not data['Date'].isin(purged_dates).any(), 'No purged XLE date may appear in the feature dataset.'
first_xle_test_date = test_data['Date'].min()
assert first_xle_test_date == first_test == pd.Timestamp('2021-03-10')
assert (purged_dates < first_xle_test_date).all()
for purged_date in purged_dates:
    purged_position = raw.index.get_loc(purged_date)
    assert (raw.index[purged_position + 1:purged_position + 6] >= first_xle_test_date).any()
assert (training_standard_deviation > 0).all(), 'Every noise scale must be positive.'
assert set().union(*[set(values) for key, values in FEATURE_GROUPS.items() if key != 'all_predictors']) == set(predictors), \
    'The four substantive groups must cover all predictors.'

print(f'Training observations: {len(X_train):,}')
print(f'Fixed-test observations: {len(X_test_clean):,}')
print(f'Feature groups: {list(FEATURE_GROUPS)}')
print(f'Noise intensities: {INTENSITIES}; random seeds: {SEEDS[0]} through {SEEDS[-1]}')
print('The 0.50 and 1.00 stress levels were added after review of the small pilot degradation; they are exploratory.')

## 2. Clean model refitting and reproduction

The established Logistic Regression pipeline and pre-specified Random Forest are refitted once using clean training observations only. Test dates are excluded from both fit calls. The clean fixed-test predictions must reproduce the saved baseline metrics before any noisy evaluation begins.

In [ ]:
def make_logistic_model():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('logistic', LogisticRegression(
            penalty='l2', C=1.0, class_weight='balanced', solver='lbfgs',
            max_iter=2000, random_state=42,
        )),
    ])

def make_random_forest():
    return RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=5,
        max_features='sqrt', class_weight='balanced_subsample',
        random_state=42, n_jobs=-1,
    )

def calculate_metrics(y_true, probability):
    prediction = (probability >= 0.5).astype(int)
    return {
        'accuracy': accuracy_score(y_true, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'f1_score': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability),
        'average_precision': average_precision_score(y_true, probability),
    }

test_dates = set(test_data['Date'])
fit_audit = []

def audited_clean_fit(model, X_fit, y_fit, dates, model_name):
    assert set(pd.DatetimeIndex(dates)).isdisjoint(test_dates), f'Test dates entered {model_name} fitting.'
    pd.testing.assert_frame_equal(X_fit, X_train, check_exact=True)
    model.fit(X_fit, y_fit)
    fit_audit.append({'model': model_name, 'fit_observations': len(X_fit), 'noisy_fit': False})
    return model

models = {
    'Logistic Regression': audited_clean_fit(
        make_logistic_model(), X_train, y_train, train_data['Date'], 'Logistic Regression'
    ),
    'Random Forest': audited_clean_fit(
        make_random_forest(), X_train, y_train, train_data['Date'], 'Random Forest'
    ),
}
assert len(models['Random Forest'].estimators_) == 500, 'The Random Forest must contain 500 fitted trees.'
assert len(fit_audit) == 2 and not any(item['noisy_fit'] for item in fit_audit), \
    'Exactly two clean fits are permitted, with no noisy-data fitting.'

clean_metrics = {}
for model_name, model in models.items():
    clean_probability = model.predict_proba(X_test_clean)[:, 1
    ]
    assert np.logical_and(clean_probability >= 0, clean_probability <= 1).all()
    clean_metrics[model_name] = calculate_metrics(y_test, clean_probability)
pd.DataFrame([{'model': k, **v} for k, v in clean_metrics.items()]).to_csv(PROJECT_ROOT / 'outputs/tables/xle_clean_baseline_metrics.csv', index=False)
print('Clean XLE baseline metrics:')
print(pd.DataFrame(clean_metrics).T[METRICS].to_string(float_format=lambda value: f'{value:.6f}'))


## 3. Paired test-time Gaussian perturbations

For one group at a time, noise equals the pre-specified intensity multiplied by the training-sample standard deviation and an independent standard-normal draw. The noisy test matrix is generated once per group, intensity and seed and then supplied unchanged to both models. Only logical domain bounds are applied: non-negative volatility, downside volatility, volume ratio and intraday range; RSI in [0, 100]; and price-to-moving-average values no lower than −1. Training-range clipping is not used.

In [ ]:
LOWER_ZERO = {
    'volatility_5d', 'volatility_20d', 'downside_volatility_20d',
    'volume_ratio_20d', 'intraday_range',
}
PRICE_TO_MA = {'price_to_ma_10', 'price_to_ma_50'}

def apply_logical_bounds(noisy_frame, selected_features):
    bounded = noisy_frame.copy()
    clipping_counts = {feature: 0 for feature in predictors}
    for feature in selected_features:
        before = bounded[feature].copy()
        if feature in LOWER_ZERO:
            bounded[feature] = bounded[feature].clip(lower=0)
        elif feature == 'rsi_14':
            bounded[feature] = bounded[feature].clip(lower=0, upper=100)
        elif feature in PRICE_TO_MA:
            bounded[feature] = bounded[feature].clip(lower=-1)
        clipping_counts[feature] = int(before.ne(bounded[feature]).sum())
    return bounded, clipping_counts

detailed_rows = []
clipping_rows = []
prediction_audit = []

for feature_group, selected_features in FEATURE_GROUPS.items():
    non_selected_features = [feature for feature in predictors if feature not in selected_features]
    for intensity in INTENSITIES:
        for seed in SEEDS:
            generator = np.random.default_rng(seed)
            standard_noise = generator.normal(size=(len(X_test_clean), len(selected_features)))
            X_noisy = X_test_clean.copy(deep=True)
            noise_scale = intensity * training_standard_deviation[selected_features].to_numpy()
            X_noisy.loc[:, selected_features] = (
                X_test_clean[selected_features].to_numpy() + standard_noise * noise_scale
            )
            X_noisy, clipping_counts = apply_logical_bounds(X_noisy, selected_features)

            if non_selected_features:
                pd.testing.assert_frame_equal(
                    X_noisy[non_selected_features], X_test_clean[non_selected_features], check_exact=True
                )
            assert np.isfinite(X_noisy.to_numpy(dtype=float)).all(), \
                'Noise must not create missing or infinite predictor values.'
            pd.testing.assert_series_equal(y_test, test_targets_before_experiment, check_exact=True)

            dataset_signature = int(pd.util.hash_pandas_object(X_noisy, index=True).sum())
            clipping_rows.append({
                'feature_group': feature_group, 'noise_intensity': intensity, 'seed': seed,
                'total_clipped_values': int(sum(clipping_counts.values())),
                **{f'clipped_{feature}': clipping_counts[feature] for feature in predictors},
            })

            for model_name, model in models.items():
                probability = model.predict_proba(X_noisy)[:, 1]
                assert np.logical_and(probability >= 0, probability <= 1).all(), \
                    f'{model_name} noisy probabilities must lie between zero and one.'
                noisy_metrics = calculate_metrics(y_test, probability)
                row = {
                    'model': model_name, 'feature_group': feature_group,
                    'noise_intensity': intensity, 'seed': seed,
                    'dataset_signature': dataset_signature,
                }
                for metric in METRICS:
                    row[f'clean_{metric}'] = clean_metrics[model_name][metric]
                    row[f'noisy_{metric}'] = noisy_metrics[metric]
                    row[f'{metric}_degradation'] = clean_metrics[model_name][metric] - noisy_metrics[metric]
                detailed_rows.append(row)
                prediction_audit.append({
                    'model': model_name, 'feature_group': feature_group,
                    'noise_intensity': intensity, 'seed': seed,
                    'dataset_signature': dataset_signature, 'object_id': id(X_noisy),
                })

detailed = pd.DataFrame(detailed_rows)
clipping_diagnostics = pd.DataFrame(clipping_rows)
prediction_audit = pd.DataFrame(prediction_audit)

expected_conditions = len(FEATURE_GROUPS) * len(INTENSITIES) * len(SEEDS)
assert len(detailed) == expected_conditions * len(models), 'Every model-condition-seed result must be present.'
assert len(clipping_diagnostics) == expected_conditions, 'Every clipping diagnostic must be present.'
assert len(detailed) == 1500, 'The extended detailed table must contain exactly 1,500 rows.'
assert len(clipping_diagnostics) == 750, 'The extended clipping table must contain exactly 750 rows.'
assert set(detailed['seed']) == set(SEEDS), 'All requested seeds must be present.'
assert set(detailed['noise_intensity']) == set(INTENSITIES), 'All requested intensities must be present.'
assert set(detailed['feature_group']) == set(FEATURE_GROUPS), 'All requested feature groups must be present.'
paired_signatures = prediction_audit.groupby(
    ['feature_group', 'noise_intensity', 'seed']
)[['dataset_signature', 'object_id']].nunique()
assert paired_signatures.eq(1).all().all(), 'Both models must receive the same noisy observations.'
assert len(fit_audit) == 2 and not any(item['noisy_fit'] for item in fit_audit), \
    'Models must not be refitted on noisy data.'
pd.testing.assert_frame_equal(data[['Date', 'sample_period']], original_dates)
pd.testing.assert_frame_equal(data[['Date', target]], original_targets)
pd.testing.assert_series_equal(y_test, test_targets_before_experiment, check_exact=True)

if existing_detailed_reference is not None:
    pilot_reference = existing_detailed_reference.loc[
        existing_detailed_reference['noise_intensity'].isin(PILOT_INTENSITIES)
    ].sort_values(['model', 'feature_group', 'noise_intensity', 'seed']).reset_index(drop=True)
    pilot_recomputed = detailed.loc[
        detailed['noise_intensity'].isin(PILOT_INTENSITIES)
    ].sort_values(['model', 'feature_group', 'noise_intensity', 'seed']).reset_index(drop=True)
    assert len(pilot_reference) == len(pilot_recomputed) == 900, 'All 900 pilot result rows must be retained.'
    pilot_reference = pilot_reference.drop(columns=['dataset_signature'])
    pilot_recomputed = pilot_recomputed.drop(columns=['dataset_signature'])
    pd.testing.assert_frame_equal(
        pilot_recomputed, pilot_reference, check_dtype=False, check_exact=False, rtol=1e-13, atol=1e-13
    )
if existing_clipping_reference is not None:
    pilot_clipping_reference = existing_clipping_reference.loc[
        existing_clipping_reference['noise_intensity'].isin(PILOT_INTENSITIES)
    ].sort_values(['feature_group', 'noise_intensity', 'seed']).reset_index(drop=True)
    pilot_clipping_recomputed = clipping_diagnostics.loc[
        clipping_diagnostics['noise_intensity'].isin(PILOT_INTENSITIES)
    ].sort_values(['feature_group', 'noise_intensity', 'seed']).reset_index(drop=True)
    assert len(pilot_clipping_reference) == len(pilot_clipping_recomputed) == 450, \
        'All 450 pilot clipping rows must be retained.'
    pd.testing.assert_frame_equal(
        pilot_clipping_recomputed, pilot_clipping_reference,
        check_dtype=False, check_exact=False, rtol=1e-13, atol=1e-13,
    )

detailed.to_csv(DETAILED_PATH, index=False)
clipping_diagnostics.to_csv(CLIPPING_PATH, index=False)
print(f'Completed {expected_conditions:,} paired noisy test conditions and {len(detailed):,} model evaluations.')
print('The original 0.05–0.20 pilot rows were reproduced and retained within the extended tables.')
current_clipping_total = int(clipping_diagnostics['total_clipped_values'].sum())
print(f'Total logical-bound clipping events: {current_clipping_total:,}')

## 4. Seed-level summary and confidence intervals

For each model, group, intensity and metric, the table reports the mean and sample standard deviation across 30 seeds. The 95% confidence interval uses the normal approximation, mean ± 1.96 × standard deviation / √30. Negative degradation is retained when perturbation improves a metric.

In [ ]:
summary_rows = []
for (model_name, feature_group, intensity), condition in detailed.groupby(
    ['model', 'feature_group', 'noise_intensity'], sort=False
):
    assert condition['seed'].nunique() == 30, 'Every summary condition must contain 30 seeds.'
    for metric in METRICS:
        performance = condition[f'noisy_{metric}']
        degradation = condition[f'{metric}_degradation']
        performance_mean = float(performance.mean())
        performance_std = float(performance.std(ddof=1))
        degradation_mean = float(degradation.mean())
        degradation_std = float(degradation.std(ddof=1))
        performance_margin = 1.96 * performance_std / np.sqrt(30)
        degradation_margin = 1.96 * degradation_std / np.sqrt(30)
        summary_rows.append({
            'model': model_name, 'feature_group': feature_group,
            'noise_intensity': intensity, 'metric': metric, 'seed_count': 30,
            'clean_performance': clean_metrics[model_name][metric],
            'mean_noisy_performance': performance_mean,
            'std_noisy_performance': performance_std,
            'noisy_performance_ci95_lower': performance_mean - performance_margin,
            'noisy_performance_ci95_upper': performance_mean + performance_margin,
            'mean_performance_degradation': degradation_mean,
            'std_performance_degradation': degradation_std,
            'degradation_ci95_lower': degradation_mean - degradation_margin,
            'degradation_ci95_upper': degradation_mean + degradation_margin,
        })
summary = pd.DataFrame(summary_rows)
assert len(summary) == len(models) * len(FEATURE_GROUPS) * len(INTENSITIES) * len(METRICS), \
    'The summary must contain every model, group, intensity and metric.'
assert len(summary) == 350, 'The extended summary table must contain exactly 350 rows.'
assert np.isfinite(summary.select_dtypes(include=np.number).to_numpy()).all(), \
    'All summary values must be finite.'
if existing_summary_reference is not None:
    pilot_summary_reference = existing_summary_reference.loc[
        existing_summary_reference['noise_intensity'].isin(PILOT_INTENSITIES)
    ].sort_values(['model', 'feature_group', 'noise_intensity', 'metric']).reset_index(drop=True)
    pilot_summary_recomputed = summary.loc[
        summary['noise_intensity'].isin(PILOT_INTENSITIES)
    ].sort_values(['model', 'feature_group', 'noise_intensity', 'metric']).reset_index(drop=True)
    assert len(pilot_summary_reference) == len(pilot_summary_recomputed) == 210, \
        'All 210 pilot summary rows must be retained.'
    pd.testing.assert_frame_equal(
        pilot_summary_recomputed, pilot_summary_reference,
        check_dtype=False, check_exact=False, rtol=1e-13, atol=1e-13,
    )
summary.to_csv(SUMMARY_PATH, index=False)
print(f'Saved {len(summary):,} model–group–intensity–metric summary rows.')

## 5. Robustness curves

Each absolute-performance and degradation figure presents one panel per model and one line per feature group. The clean score or zero degradation at zero noise is the common reference. Shaded bands show the 95% confidence interval across seeds. Metric-specific y-axis limits are narrowed to the observed confidence-interval range, with a small margin, because forcing a 0–1 scale would conceal the relatively small differences. Both model panels use identical limits within each metric. Negative degradation is retained and is not interpreted as a genuine improvement unless its confidence interval excludes zero and the magnitude is substantively meaningful.

In [ ]:
METRIC_TITLES = {
    'balanced_accuracy': 'Balanced Accuracy',
    'f1_score': 'F1-Score',
    'roc_auc': 'ROC-AUC',
    'average_precision': 'Average Precision',
}
GROUP_LABELS = {
    'returns': 'Returns', 'volatility': 'Volatility', 'momentum': 'Momentum',
    'volume_and_range': 'Volume and range', 'all_predictors': 'All predictors',
}

def limits_with_margin(minimum, maximum):
    span = maximum - minimum
    margin = max(0.08 * span, 0.001)
    return minimum - margin, maximum + margin

def plot_robustness_metric(metric):
    metric_rows = summary.loc[summary['metric'].eq(metric)]
    observed_minimum = min(
        metric_rows['noisy_performance_ci95_lower'].min(),
        min(clean_metrics[model_name][metric] for model_name in models),
    )
    observed_maximum = max(
        metric_rows['noisy_performance_ci95_upper'].max(),
        max(clean_metrics[model_name][metric] for model_name in models),
    )
    y_limits = limits_with_margin(observed_minimum, observed_maximum)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)
    for ax, model_name in zip(axes, models):
        for feature_group in FEATURE_GROUPS:
            rows = summary.loc[
                summary['model'].eq(model_name)
                & summary['feature_group'].eq(feature_group)
                & summary['metric'].eq(metric)
            ].sort_values('noise_intensity')
            x_values = np.array([0.0, *rows['noise_intensity'].to_numpy(dtype=float)])
            clean_score = clean_metrics[model_name][metric]
            mean_values = np.array([clean_score, *rows['mean_noisy_performance'].to_numpy(dtype=float)])
            lower_values = np.array([clean_score, *rows['noisy_performance_ci95_lower'].to_numpy(dtype=float)])
            upper_values = np.array([clean_score, *rows['noisy_performance_ci95_upper'].to_numpy(dtype=float)])
            line = ax.plot(x_values, mean_values, marker='o', linewidth=2, label=GROUP_LABELS[feature_group])[0]
            ax.fill_between(x_values, lower_values, upper_values, color=line.get_color(), alpha=0.15)
        ax.set_title(model_name)
        ax.set_xlabel('Noise intensity')
        ax.set_ylabel(METRIC_TITLES[metric])
        ax.set_xlim(0, max(INTENSITIES))
        ax.set_ylim(*y_limits)
    axes[-1].legend(title='Perturbed feature group', loc='best')
    fig.suptitle(f'XLE Test-Time Noise Robustness: {METRIC_TITLES[metric]}')
    fig.text(
        0.5, 0.01,
        'Narrowed y-axis limits show relatively small differences and their confidence bands.',
        ha='center', fontsize=9,
    )
    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    fig.savefig(FIGURE_PATHS[metric], dpi=300, bbox_inches='tight')
    plt.show()

def plot_degradation_metric(metric):
    metric_rows = summary.loc[summary['metric'].eq(metric)]
    observed_minimum = min(0.0, metric_rows['degradation_ci95_lower'].min())
    observed_maximum = max(0.0, metric_rows['degradation_ci95_upper'].max())
    y_limits = limits_with_margin(observed_minimum, observed_maximum)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)
    for ax, model_name in zip(axes, models):
        for feature_group in FEATURE_GROUPS:
            rows = summary.loc[
                summary['model'].eq(model_name)
                & summary['feature_group'].eq(feature_group)
                & summary['metric'].eq(metric)
            ].sort_values('noise_intensity')
            x_values = np.array([0.0, *rows['noise_intensity'].to_numpy(dtype=float)])
            mean_values = np.array([0.0, *rows['mean_performance_degradation'].to_numpy(dtype=float)])
            lower_values = np.array([0.0, *rows['degradation_ci95_lower'].to_numpy(dtype=float)])
            upper_values = np.array([0.0, *rows['degradation_ci95_upper'].to_numpy(dtype=float)])
            line = ax.plot(x_values, mean_values, marker='o', linewidth=2, label=GROUP_LABELS[feature_group])[0]
            ax.fill_between(x_values, lower_values, upper_values, color=line.get_color(), alpha=0.18)
        ax.axhline(0, color='black', linestyle='--', linewidth=1, label='Zero degradation')
        ax.set_title(model_name)
        ax.set_xlabel('Noise intensity')
        ax.set_ylabel(f'{METRIC_TITLES[metric]} degradation')
        ax.set_xlim(0, max(INTENSITIES))
        ax.set_ylim(*y_limits)
    axes[-1].legend(title='Perturbed feature group', loc='best')
    fig.suptitle(f'XLE Test-Time Noise Degradation: {METRIC_TITLES[metric]}')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(DEGRADATION_FIGURE_PATHS[metric], dpi=300, bbox_inches='tight')
    plt.show()

for metric in FIGURE_PATHS:
    plot_robustness_metric(metric)
    plot_degradation_metric(metric)

heatmap_data = summary.loc[summary['noise_intensity'].eq(1.00)].pivot_table(
    index=['model', 'metric'], columns='feature_group', values='mean_performance_degradation'
).reindex(
    pd.MultiIndex.from_product([list(models), METRICS], names=['model', 'metric'])
).reindex(columns=list(FEATURE_GROUPS))
heatmap_data.index = [f'{model_name} — {metric}' for model_name, metric in heatmap_data.index]
heatmap_data.columns = [GROUP_LABELS[column] for column in heatmap_data.columns]
heatmap_limit = float(np.abs(heatmap_data.to_numpy()).max())
fig, ax = plt.subplots(figsize=(11, 10))
sns.heatmap(
    heatmap_data, cmap='vlag', center=0, vmin=-heatmap_limit, vmax=heatmap_limit,
    annot=True, fmt='.3f', linewidths=0.4, cbar_kws={'label': 'Mean performance degradation'}, ax=ax,
)
ax.set_title('XLE Mean Performance Degradation at Noise Intensity 1.00')
ax.set_xlabel('Perturbed feature group')
ax.set_ylabel('Model and metric')
fig.tight_layout()
fig.savefig(HEATMAP_PATH, dpi=300, bbox_inches='tight')
plt.show()
print('Saved four zoomed absolute-performance figures, four degradation figures and one stress-test heatmap.')

## 6. Concise robustness summary

Lower mean degradation indicates greater robustness. The following diagnostics aggregate carefully for concise reporting while retaining all metric-specific results in the saved tables.

In [ ]:
group_metric_degradation = summary.groupby(
    ['model', 'feature_group', 'metric'], as_index=False
)['mean_performance_degradation'].mean()
largest_rows = group_metric_degradation.loc[
    group_metric_degradation.groupby(['model', 'metric'])['mean_performance_degradation'].idxmax()
].sort_values(['model', 'metric'])

condition_metric = summary.pivot_table(
    index=['feature_group', 'noise_intensity', 'metric'],
    columns='model', values='mean_performance_degradation'
).reset_index()
condition_metric['more_robust_model'] = np.where(
    condition_metric['Logistic Regression'] < condition_metric['Random Forest'],
    'Logistic Regression',
    np.where(
        condition_metric['Random Forest'] < condition_metric['Logistic Regression'],
        'Random Forest', 'Tie',
    ),
)
condition_overall = summary.groupby(
    ['model', 'feature_group', 'noise_intensity'], as_index=False
)['mean_performance_degradation'].mean().pivot_table(
    index=['feature_group', 'noise_intensity'], columns='model', values='mean_performance_degradation'
).reset_index()
condition_overall['more_robust_model_on_average'] = np.where(
    condition_overall['Logistic Regression'] < condition_overall['Random Forest'],
    'Logistic Regression', 'Random Forest',
)
ranking_variation = condition_metric.groupby(
    ['feature_group', 'noise_intensity']
)['more_robust_model'].nunique().rename('distinct_metric_rankings').reset_index()
condition_overall = condition_overall.merge(
    ranking_variation, on=['feature_group', 'noise_intensity'], how='left', validate='one_to_one'
)
condition_overall['ranking_changes_across_metrics'] = condition_overall['distinct_metric_rankings'] > 1

clipping_by_group = clipping_diagnostics.groupby('feature_group')['total_clipped_values'].sum().sort_values(ascending=False)
total_clipping_events = int(clipping_diagnostics['total_clipped_values'].sum())

PRIMARY_METRICS = ['balanced_accuracy', 'f1_score', 'roc_auc', 'average_precision']
all_predictors_results = summary.loc[
    summary['feature_group'].eq('all_predictors') & summary['metric'].isin(PRIMARY_METRICS),
    [
        'model', 'noise_intensity', 'metric', 'clean_performance',
        'mean_noisy_performance', 'mean_performance_degradation',
        'degradation_ci95_lower', 'degradation_ci95_upper',
    ],
].sort_values(['noise_intensity', 'metric', 'model'])

monotonic_rows = []
for (model_name, feature_group, metric), rows in summary.groupby(['model', 'feature_group', 'metric']):
    ordered_degradation = rows.sort_values('noise_intensity')['mean_performance_degradation'].to_numpy()
    monotonic_rows.append({
        'model': model_name, 'feature_group': feature_group, 'metric': metric,
        'monotonic_non_decreasing_degradation': bool((np.diff(ordered_degradation) >= -1e-12).all()),
    })
monotonicity = pd.DataFrame(monotonic_rows)

stress_all_predictors = condition_metric.loc[
    condition_metric['feature_group'].eq('all_predictors')
    & condition_metric['noise_intensity'].eq(1.00),
    ['metric', 'Logistic Regression', 'Random Forest', 'more_robust_model'],
].sort_values('metric')
stress_clipping = clipping_diagnostics.loc[
    clipping_diagnostics['noise_intensity'].isin(STRESS_TEST_INTENSITIES)
].groupby(['noise_intensity', 'feature_group'])['total_clipped_values'].sum().unstack(fill_value=0)

detectable = (summary['degradation_ci95_lower'] > 0) | (summary['degradation_ci95_upper'] < 0)
detectable_count = int(detectable.sum())
maximum_absolute_mean_degradation = float(summary['mean_performance_degradation'].abs().max())
most_damaging_overall = summary.groupby(
    ['model', 'feature_group'], as_index=False
)['mean_performance_degradation'].mean().loc[
    lambda frame: frame.groupby('model')['mean_performance_degradation'].idxmax()
].sort_values('model')

print('Feature group with the largest mean degradation, averaged across intensities, for each model and metric:')
print(largest_rows.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('All-predictors clean, noisy and degradation results:')
print(all_predictors_results.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
monotonic_count = int(monotonicity['monotonic_non_decreasing_degradation'].sum())
print(f'Monotonic degradation sequences: {monotonic_count} of {len(monotonicity)} model–group–metric sequences.')
print('Non-monotonic model–group–metric sequences:')
print(monotonicity.loc[
    ~monotonicity['monotonic_non_decreasing_degradation'],
    ['model', 'feature_group', 'metric'],
].to_string(index=False))
print('Most damaging feature group for each model, averaged across intensities and metrics:')
print(most_damaging_overall.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('All-predictors stress-test robustness at intensity 1.00; lower degradation is more robust:')
print(stress_all_predictors.to_string(index=False, float_format=lambda value: f'{value:.6f}'))
print('More robust model under each group and intensity, averaging the seven degradation metrics:')
print(condition_overall[[
    'feature_group', 'noise_intensity', 'more_robust_model_on_average',
    'ranking_changes_across_metrics',
]].to_string(index=False))
ranking_change_count = int(condition_overall['ranking_changes_across_metrics'].sum())
print(f'Conditions whose model ranking changes across metrics: {ranking_change_count} of {len(condition_overall)}')
print(f'Total logical-bound clipping events: {total_clipping_events:,}')
print('Clipping events by feature group:')
print(clipping_by_group.to_string())
print('Clipping events at stress-test intensities 0.50 and 1.00:')
print(stress_clipping.to_string())
print(
    f'{detectable_count} of {len(summary)} confidence intervals exclude zero degradation; '
    f'the largest absolute mean degradation is {maximum_absolute_mean_degradation:.6f}. '
    'Confidence-interval exclusion indicates statistical detectability, whereas substantive importance '
    'depends on the magnitude and practical forecasting consequences.'
)
print(
    'Negative degradation is retained but is not described as a genuine improvement unless its confidence '
    'interval excludes zero and its magnitude is substantively meaningful.'
)

clean_after = pd.read_csv(CLEAN_METRICS_PATH)
detailed_after = pd.read_csv(DETAILED_PATH)
summary_after = pd.read_csv(SUMMARY_PATH)
clipping_after = pd.read_csv(CLIPPING_PATH)
comparison_after = pd.read_csv(COMPARISON_PATH)
assert len(clean_after) == 2
assert len(summary_after) == 350 and len(detailed_after) == 1500 and len(clipping_after) == 750
assert len(comparison_after) == 96
pd.testing.assert_frame_equal(clean_after, existing_clean_reference, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(summary_after, existing_summary_reference, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(detailed_after.drop(columns=['dataset_signature']), existing_detailed_reference.drop(columns=['dataset_signature']), check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(clipping_after, existing_clipping_reference, check_exact=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(comparison_after, existing_comparison_reference, check_exact=False, rtol=1e-12, atol=1e-12)
print('Five actual XLE purged dates:', ', '.join(date.strftime('%Y-%m-%d') for date in purged_dates))
print(f'First XLE test date: {first_xle_test_date:%Y-%m-%d}')
print('Combined clean XLE metrics:')
print(clean_after.to_string(index=False))
print('All pre-run and post-run numerical results are unchanged within floating-point tolerance.')

created_files = [
    Path('notebooks/06_xle_cross_sector_replication.ipynb'),
    Path('data/raw/xle_daily_2000_2026.csv'), Path('data/processed/xle_feature_dataset.csv'),
    Path('outputs/tables/xle_clean_baseline_metrics.csv'),
    Path('outputs/tables/xlk_xle_key_results_comparison.csv'),
    DETAILED_PATH.relative_to(PROJECT_ROOT), SUMMARY_PATH.relative_to(PROJECT_ROOT),
    CLIPPING_PATH.relative_to(PROJECT_ROOT),
    *[path.relative_to(PROJECT_ROOT) for path in FIGURE_PATHS.values()],
    *[path.relative_to(PROJECT_ROOT) for path in DEGRADATION_FIGURE_PATHS.values()],
    HEATMAP_PATH.relative_to(PROJECT_ROOT),
]
print('Files created or modified by this experiment:')
for path in created_files:
    print(f'- {path}')
print('No model was tuned or refitted on noisy data, and no target or earlier project file was modified.')

## Limitations

XLE reproduces the XLK methodology but estimates its target threshold and feature-noise scales independently from XLE training data. The 0.05–0.20 results are the primary moderate-noise evidence. Intensities 0.50 and 1.00 are exploratory stress tests, and extensive logical-bound clipping at these levels means that they represent bounded-input stress scenarios rather than unconstrained Gaussian perturbations. The perturbations are synthetic and do not establish the empirical distribution of real data errors. Robustness rankings can differ by metric, feature group and noise intensity; no single aggregate ranking should replace the complete results. Statistical detectability does not by itself establish a substantively important effect size.